# Spatial-ring sector interface: fail-closed Colab verification

This notebook verifies the **WIP/preregistered** Lean interface at one immutable public SHA. It is not a certificate by itself. It fails before compilation if the repository SHA, Lean toolchain, or Mathlib pin differs from the registered values. A final `PASS` is printed only after the module build, the complete new-declaration axiom oracle, the consistency judge, and `lake build YangMillsCore` all succeed.

The target remains: for `beta >= 0`, `gamma >= 0`, every extent, `specRatio <= tanh(beta) * exp(2*gamma)`. This notebook validates only the abstract sector-interface milestone; it does not prove either analytic sector bound.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, re, shutil, subprocess, tempfile

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
EXPECTED_SHA = 'd2966688d52bb0217f2165f544d0b95e5749a4d9'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
RUN_ROOT = Path(tempfile.mkdtemp(prefix='spatial-ring-sector-'))
REPO = RUN_ROOT / 'repo'
ARTIFACTS = RUN_ROOT / 'artifacts'
ARTIFACTS.mkdir()
TRANSCRIPT = ARTIFACTS / 'transcript.txt'

def log(text):
    text = str(text)
    print(text)
    with TRANSCRIPT.open('a', encoding='utf-8', newline='\n') as f:
        f.write(text + '\n')

def run(cmd, cwd=None, env=None, allow_failure=False):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    try:
        p = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    except FileNotFoundError as exc:
        log(f'[missing executable: {exc.filename}]')
        log('[exit 127]')
        if allow_failure:
            return subprocess.CompletedProcess(cmd, 127, stdout=str(exc))
        raise
    log(p.stdout.rstrip())
    log(f'[exit {p.returncode}]')
    if p.returncode and not allow_failure:
        raise RuntimeError(f'command failed ({p.returncode}): {shown}')
    return p

log('SPATIAL-RING SECTOR INTERFACE COLAB RUN')
log(f'utc_start={datetime.datetime.now(datetime.timezone.utc).isoformat()}')
log(f'platform={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').splitlines()[4] if Path('/proc/cpuinfo').exists() else 'cpuinfo=unavailable')
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0] if Path('/proc/meminfo').exists() else 'meminfo=unavailable')
run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], allow_failure=True)
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
actual_sha = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip()
if actual_sha != EXPECTED_SHA:
    raise RuntimeError(f'SHA mismatch: {actual_sha} != {EXPECTED_SHA}')
toolchain = (REPO / 'lean-toolchain').read_text(encoding='utf-8').strip()
if toolchain != EXPECTED_TOOLCHAIN:
    raise RuntimeError(f'toolchain mismatch: {toolchain} != {EXPECTED_TOOLCHAIN}')
manifest = json.loads((REPO / 'lake-manifest.json').read_text(encoding='utf-8'))
mathlib_entries = [p for p in manifest['packages'] if p.get('name') == 'mathlib']
if len(mathlib_entries) != 1:
    raise RuntimeError(f'expected exactly one mathlib manifest entry, got {len(mathlib_entries)}')
manifest_rev = mathlib_entries[0].get('rev')
if manifest_rev != EXPECTED_MATHLIB:
    raise RuntimeError(f'mathlib mismatch: {manifest_rev} != {EXPECTED_MATHLIB}')
log(f'repo_sha={actual_sha}')
log(f'lean_toolchain={toolchain}')
log(f'mathlib_pin={manifest_rev}')
log('PRECHECK PASS')

In [ ]:
elan_home = RUN_ROOT / 'elan'
env = os.environ.copy()
env['ELAN_HOME'] = str(elan_home)
env['PATH'] = str(elan_home / 'bin') + os.pathsep + env['PATH']
installer = RUN_ROOT / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh', '-o', str(installer)])
installer_sha = hashlib.sha256(installer.read_bytes()).hexdigest()
log(f'elan_installer_sha256={installer_sha}')
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'], env=env)
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
run(['lean', '--version'], cwd=REPO, env=env)
run(['lake', '--version'], cwd=REPO, env=env)
# Official Mathlib cache is permitted only inside this isolated ephemeral runtime.
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
log('TOOLCHAIN_AND_CACHE PASS')

In [ ]:
new_declarations = [
  'flipObs_flip', 'IsFlipEven', 'IsFlipOdd', 'isFlipEven_iff', 'isFlipOdd_iff',
  'evenPart', 'oddPart', 'evenPart_add_oddPart', 'isFlipEven_evenPart',
  'isFlipOdd_oddPart', 'flipEven_flipOdd_orthogonal', 'evenPart_oddPart_orthogonal',
  'eucNorm_add_sq_of_orthogonal', 'evenPart_oddPart_norm_sq',
  'act_preserves_flipEven', 'act_preserves_flipOdd', 'flipOdd_perp_of_flipEven',
  'evenPart_perp_of_perp', 'norm_act_le_of_flip_sector_bounds',
  'act_ring_preserves_flipEven', 'act_ring_preserves_flipOdd',
  'SpatialRingOddSectorBound', 'SpatialRingEvenFluctuationBound',
  'spatialRing_specRatio_le_of_sector_bounds'
]
oracle = REPO / 'oracle_check.lean'
run(['lake', 'build', 'YangMills.OS.SpatialRing'], cwd=REPO, env=env)
core = run(['lake', 'build', 'YangMillsCore'], cwd=REPO, env=env)
match = re.search(r'Build completed successfully \((\d+) jobs\)', core.stdout)
if not match:
    raise RuntimeError('full core build succeeded without a measured job count')
jobs = int(match.group(1))
oracle_run = run(['lake', 'env', 'lean', str(oracle)], cwd=REPO, env=env)
for name in new_declarations:
    marker = f"'YangMills.OS.{name}' depends on axioms:"
    if marker not in oracle_run.stdout:
        raise RuntimeError(f'permanent oracle omitted output for {name}')
allowed_axioms = {'propext', 'Classical.choice', 'Quot.sound'}
if 'sorryAx' in oracle_run.stdout:
    raise RuntimeError('oracle contains sorryAx')
for line in oracle_run.stdout.splitlines():
    if 'depends on axioms:' not in line:
        continue
    payload = line.split('depends on axioms:', 1)[1].strip().strip('[]')
    used = {x.strip() for x in payload.split(',') if x.strip()}
    if not used <= allowed_axioms:
        raise RuntimeError(f'nonstandard axioms: {used - allowed_axioms}')
run(['python3', 'scripts/check_consistency.py'], cwd=REPO, env=env)
probe_judge = RUN_ROOT / 'judge_spatial_sector_probe.py'
probe_judge_url = ('https://raw.githubusercontent.com/lluiseriksson/'
    'THE-ERIKSSON-PROGRAMME/e8e0725251b291a8a3e035ebd534e9003300896b/'
    'scripts/judge_spatial_sector_probe.py')
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', probe_judge_url,
    '-o', str(probe_judge)])
run(['python3', str(probe_judge)], cwd=REPO, env=env)
metadata = {
  'repo_sha': actual_sha, 'toolchain': toolchain, 'mathlib_pin': manifest_rev,
  'jobs': jobs, 'utc_end': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'cpu_count': os.cpu_count(),
  'elan_installer_sha256': installer_sha,
}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
hashes = {}
for path in sorted(ARTIFACTS.iterdir()):
    if path.name != 'SHA256SUMS':
        hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
(ARTIFACTS / 'SHA256SUMS').write_text(''.join(f'{h}  {name}\n' for name, h in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_ring_sector_artifacts', 'zip', ARTIFACTS)
log(f'jobs_measured={jobs}')
log(f'artifact_zip={archive}')
log(f'artifact_zip_sha256={hashlib.sha256(Path(archive).read_bytes()).hexdigest()}')
log('SPATIAL-RING SECTOR INTERFACE PASS')
from google.colab import files
files.download(archive)